In [1]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import  Ridge, Lasso

df = pd.read_csv('beer-servings.csv')
df.columns.tolist()

['Unnamed: 0',
 'country',
 'beer_servings',
 'spirit_servings',
 'wine_servings',
 'total_litres_of_pure_alcohol',
 'continent']

In [2]:
df=df.drop(['Unnamed: 0'], axis=1)
df.drop_duplicates(inplace=True)
df= df.drop(columns = ["country", "continent"])
df.columns.tolist()

['beer_servings',
 'spirit_servings',
 'wine_servings',
 'total_litres_of_pure_alcohol']

In [3]:
cols = df.columns.tolist()

for col in cols:
    df[col]= df[col].fillna(df[col].median())
    df.isna().sum()

In [7]:
def remove_outliers(df,column_name):
    q1= df[column_name].quantile(0.25)
    q3= df[column_name].quantile(0.75)
    iqr=q3-q1
    upper_bound = q3+1.5*iqr
    lower_bound = q1-1.5*iqr
    df[column_name]= df[column_name].clip(upper=upper_bound)
    df[column_name]= df[column_name].clip(lower=lower_bound)
    return df[column_name]

In [9]:
for col in cols:
    df[col]=remove_outliers(df,col)

In [11]:
y = df['total_litres_of_pure_alcohol']
X = df.drop('total_litres_of_pure_alcohol', axis = 1)

X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2)

##  Polynomial Regression

In [14]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

poly = PolynomialFeatures(degree=3, include_bias=False)
X_train_poly = poly.fit_transform(X_train_scaled)
X_test_poly = poly.transform(X_test_scaled)

model = LinearRegression()
model.fit(X_train_poly, y_train)

y_train_pred = model.predict(X_train_poly)
y_test_pred = model.predict(X_test_poly)

poly_mse_train = mean_squared_error(y_train, y_train_pred)
poly_r2_train = r2_score(y_train, y_train_pred)

poly_mse_test = mean_squared_error(y_test, y_test_pred)
poly_r2_test = r2_score(y_test, y_test_pred)


print(f"Train MSE: {poly_mse_train:.2f}")
print(f"Train R-squared: {poly_r2_train:.2f}")

print(f"Test MSE: {poly_mse_test:.2f}")
print(f"Test R-squared: {poly_r2_test:.2f}")

Train MSE: 1.81
Train R-squared: 0.87
Test MSE: 1.89
Test R-squared: 0.88


##   Linaer Regression

In [17]:
# scaler = StandardScaler()
# X_train_scaled = scaler.fit_transform(X_train)
# X_test_scaled = scaler.transform(X_test)


model = LinearRegression()
model.fit(X_train_scaled, y_train)


y_train_pred = model.predict(X_train_scaled)
y_test_pred = model.predict(X_test_scaled)

# Evaluation
lin_mse_train = mean_squared_error(y_train, y_train_pred)
lin_r2_train = r2_score(y_train, y_train_pred)

lin_mse_test = mean_squared_error(y_test, y_test_pred)
lin_r2_test = r2_score(y_test, y_test_pred)

print(f"Train MSE: {lin_mse_train:.2f}")
print(f"Train R-squared: {lin_r2_train:.2f}")
print(f"Test MSE: {lin_mse_test:.2f}")
print(f"Train R-squared: {lin_r2_test:.2f}")



Train MSE: 2.13
Train R-squared: 0.84
Test MSE: 2.08
Train R-squared: 0.86


## Ridge

In [20]:
ridge = Ridge(alpha=1.0) 
ridge.fit(X_train_scaled, y_train)


y_train_pred = ridge.predict(X_train_scaled)
y_test_pred = ridge.predict(X_test_scaled)

rid_mse_train = mean_squared_error(y_train, y_train_pred)
rid_r2_train = r2_score(y_train, y_train_pred)

rid_mse_test = mean_squared_error(y_test, y_test_pred)
rid_r2_test = r2_score(y_test, y_test_pred)

print(f"Train MSE: {rid_mse_train:.2f}")
print(f"Train R-squared: {rid_r2_train:.2f}")
print(f"Test MSE: {rid_mse_test:.2f}")
print(f"Test R-squared: {rid_r2_test:.2f}")


Train MSE: 2.13
Train R-squared: 0.84
Test MSE: 2.09
Test R-squared: 0.86


## Lasso

In [23]:
lasso = Lasso(alpha=0.1)
lasso.fit(X_train_scaled, y_train)


y_train_pred = lasso.predict(X_train_scaled)
y_test_pred = lasso.predict(X_test_scaled)

# Evaluation
lass_mse_train = mean_squared_error(y_train, y_train_pred)
lass_r2_train = r2_score(y_train, y_train_pred)

lass_mse_test = mean_squared_error(y_test, y_test_pred)
lass_r2_test = r2_score(y_test, y_test_pred)

print("Lasso Regression")
print(f"Train MSE: {lass_mse_train:.2f}")
print(f"Train R-squared: {lass_r2_train:.2f}")
print(f"Test MSE: {lass_mse_test:.2f}")
print(f"Test R-squared: {lass_r2_test:.2f}")

Lasso Regression
Train MSE: 2.15
Train R-squared: 0.84
Test MSE: 2.17
Test R-squared: 0.86


In [25]:
result_df = pd.DataFrame(columns=["Model", "R2-Score (Train)", "R2-Score (Test)"])
result_df.loc[len(result_df)] = ["Linear Regression", lin_r2_train, lin_r2_test]
result_df.loc[len(result_df)] = ["Ridge Regression", rid_r2_train, rid_r2_test]
result_df.loc[len(result_df)] = ["Lasso Regression", lass_r2_train, lass_r2_test]
result_df.loc[len(result_df)] = ["Polynomial Regression", poly_r2_train, poly_r2_test]
result_df

,Model,R2-Score (Train),R2-Score (Test)
0,Linear Regression,0.844677,0.864363
1,Ridge Regression,0.844667,0.864109
2,Lasso Regression,0.843491,0.858823
3,Polynomial Regression,0.867645,0.877077
